# Device Financing Credit Risk Modeling

**Objective.** Build an interpretable default-risk baseline, evaluate it on future applications, and translate model scores into an approval strategy.

> This notebook uses an instructor-simulated academic dataset. It is not affiliated with Verizon and does not contain internal Verizon customer data. The raw course file is intentionally excluded from the public repository.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.credit_risk import (
    MODEL_FEATURES,
    build_models,
    evaluate_model,
    load_data,
    logistic_coefficients,
    temporal_split,
    threshold_strategy,
)

DATA_PATH = ROOT / "data" / "raw" / "verizon_data.csv"
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Data contract and portfolio profile

In [2]:
data = load_data(DATA_PATH)

profile = pd.Series({
    "Applications": len(data),
    "Date start": data["application_date"].min().date(),
    "Date end": data["application_date"].max().date(),
    "Missing values": int(data.isna().sum().sum()),
    "Duplicate rows": int(data.duplicated().sum()),
    "Overall default rate": f"{data['target'].mean():.1%}",
})
profile.to_frame("Value")

,Value
Applications,12000
Date start,2020-01-01
Date end,2025-09-30
Missing values,0
Duplicate rows,0
Overall default rate,24.6%


The source has seven fields. The pipeline derives `dp_ratio`, `financed_amount`, and a binary target. Validation occurs before any modeling step.

In [3]:
annual_default = (
    data.assign(year=data["application_date"].dt.year)
    .groupby("year", as_index=False)
    .agg(applications=("target", "size"), default_rate=("target", "mean"))
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
data["target"].value_counts().sort_index().plot.bar(
    ax=axes[0], color=["#0B5963", "#D97706"]
)
axes[0].set_title("Class balance")
axes[0].set_xlabel("0 = performing, 1 = 90+ delinquent/default")
axes[0].set_ylabel("Applications")

axes[1].bar(annual_default["year"].astype(str), annual_default["default_rate"], color="#0B5963")
axes[1].set_title("Default prevalence changes over time")
axes[1].set_xlabel("Application year")
axes[1].set_ylabel("Default rate")
axes[1].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
plt.tight_layout()
plt.show()

annual_default.style.format({"default_rate": "{:.1%}"})

/var/folders/qw/v1mp9h4n1f987rsx0zff8yvc0000gn/T/ipykernel_24252/2731460883.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,year,applications,default_rate
0,2020,2128,28.7%
1,2021,2027,28.3%
2,2022,2080,24.5%
3,2023,2076,26.5%
4,2024,2092,23.7%
5,2025,1597,13.3%


**Risk observation.** The target rate falls materially in 2025. This makes a time-based holdout essential and creates a calibration risk that a random train/test split would hide.

## 2. Strict out-of-time validation

In [4]:
split = temporal_split(data, cutoff="2025-01-01")

pd.DataFrame({
    "Sample": ["Development", "Out-of-time test"],
    "Rows": [len(split.train), len(split.test)],
    "Start": [split.train.application_date.min().date(), split.test.application_date.min().date()],
    "End": [split.train.application_date.max().date(), split.test.application_date.max().date()],
    "Default rate": [split.train.target.mean(), split.test.target.mean()],
}).style.format({"Default rate": "{:.1%}"})

,Sample,Rows,Start,End,Default rate
0,Development,10403,2020-01-01,2024-12-31,26.3%
1,Out-of-time test,1597,2025-01-01,2025-09-30,13.3%


Numeric preprocessing uses training-only 1st/99th percentile clipping and standardization. Payment type is one-hot encoded. The OOT sample is never used to fit preprocessing or model parameters.

## 3. Model comparison

In [5]:
models = build_models()
metrics_rows = []
probabilities = {}

for name, model in models.items():
    model.fit(split.train[MODEL_FEATURES], split.train["target"])
    metrics, probability = evaluate_model(
        model, split.test[MODEL_FEATURES], split.test["target"]
    )
    metrics_rows.append({"model": name, **metrics})
    probabilities[name] = probability

metrics = pd.DataFrame(metrics_rows).set_index("model")
metrics[["roc_auc", "pr_auc", "brier_score", "ks_statistic"]].style.format("{:.3f}")

,roc_auc,pr_auc,brier_score,ks_statistic
model,,,,
logistic,0.590,0.186,0.133,0.159
logistic_balanced,0.590,0.186,0.249,0.154
random_forest,0.579,0.188,0.242,0.141


![ROC and precision-recall comparison](../docs/model_comparison.png)

The ordinary logistic model has the strongest OOT ROC-AUC and substantially better Brier score than the balanced alternatives. The conventional 0.50 cutoff predicts no defaults because probabilities shift downward with the 2025 prevalence change; business thresholds must therefore be validated separately.

## 4. Interpretability

In [6]:
coefficient_table = logistic_coefficients(models["logistic"])
coefficient_table.style.format({"coefficient": "{:.3f}", "odds_ratio": "{:.3f}"})

,feature,coefficient,odds_ratio
0,intercept,-0.976,nan
1,fico,-0.254,0.775
2,borrower_age,-0.099,0.906
3,device_cost,0.020,1.020
4,down_payment,-0.013,0.987
5,dp_ratio,-0.052,0.949
6,payment_type_Cash,0.086,1.090
7,payment_type_Credit,-0.197,0.821


![Standardized logistic coefficients](../docs/logistic_coefficients.png)

Higher FICO and borrower age are associated with lower modeled risk in this simulated sample. Because numeric features are standardized, the coefficient odds ratios represent a one-standard-deviation change after clipping. These are associations, not causal effects.

## 5. Approval strategy under explicit assumptions

In [7]:
strategy = threshold_strategy(
    probabilities["logistic"],
    split.test["target"],
    split.test["financed_amount"],
    performing_margin_rate=0.12,
    loss_given_default=0.70,
)
best = strategy.loc[strategy["estimated_total_value"].idxmax()]

pd.Series({
    "Illustrative PD threshold": best["pd_threshold"],
    "Approval rate": best["approval_rate"],
    "Default rate among approved": best["default_rate_approved"],
    "Estimated total value": best["estimated_total_value"],
}).to_frame("Value")

,Value
Illustrative PD threshold,0.300000
Approval rate,0.705072
Default rate among approved,0.109236
Estimated total value,27493.976000


![Approval-risk trade-off](../docs/business_strategy.png)

The 12% performing margin and 70% loss-given-default values are transparent scenario inputs, not company estimates. A production decision requires actual revenue, loss, recovery, servicing cost, capacity, and fairness constraints.

## 6. Conclusion and limitations

The logistic model is the preferred baseline because it matches or exceeds the benchmark models on OOT discrimination while remaining easier to explain. Its predictive power is limited, so the result should support manual review and data collection rather than automatic declines.

Priority next inputs are payment history, income and employment, customer tenure, device and recovery information, and rejected applications. The 2025 prevalence shift also requires calibration and drift monitoring. Protected attributes are absent, but fairness testing would still be mandatory before operational use.